# ResikIn Waste Classifier - Model Fine-Tuning

Notebook ini digunakan untuk melatih (fine-tune) model CLIP agar lebih pintar mendeteksi sampah. Kita akan:
1. Mengunduh dataset dari Roboflow langsung ke server Google Colab.
2. Menjalankan skrip persiapan data.
3. Menjalankan proses training.

## 1. Persiapan Lingkungan (Install Dependencies)

In [ ]:
!pip install -q roboflow transformers torch torchvision scikit-learn pillow matplotlib tqdm

## 2. Unduh Dataset dari Roboflow

Sesuai dengan API yang Anda berikan, ini akan mengunduh dataset ke dalam folder `dataset_mentah` di Colab.

In [ ]:
import os
from roboflow import Roboflow

# Buat folder untuk dataset mentah
os.makedirs("dataset_mentah", exist_ok=True)
os.chdir("dataset_mentah")

# Download dari Roboflow menggunakan key Anda
rf = Roboflow(api_key="CsZFCzVXJL8qqYDhx0hR")
project = rf.workspace("project-ia-andzk").project("classification-image-6zihm")
version = project.version(2)
dataset = version.download("folder")

print(f"\nDataset berhasil diunduh di path: {dataset.location}")
os.chdir("..")

## 3. Clone Repository Anda

Kita perlu mengunduh skrip `train.py` dan `prepare_dataset.py` yang sudah dibuat. Pastikan Anda sudah me-*push* kode ke repositori GitHub Anda.

In [ ]:
# Ganti URL ini dengan URL repository GitHub baru Anda yang berisi resikin-waste-classifier
# Jika repo private, hapus baris ini dan unggah file secara manual.
!git clone https://github.com/Hanafi-Sh/resikin-waste-classifier.git repo_ai

# Jika Anda belum push repo baru, Anda bisa mengunggah folder resikin-waste-classifier 
# secara manual ke Colab dan mengubah path di sel-sel berikutnya.

## 4. Rapikan Dataset

Roboflow mengunduh dataset dengan nama folder tertentu. Kita akan menggunakan skrip `prepare_dataset.py` untuk memindahkannya ke dalam struktur yang diharapkan oleh skrip training.

In [ ]:
import glob

# Mencari folder hasil download (nama foldernya biasanya sama dengan nama project)
download_dirs = glob.glob("dataset_mentah/*/")
roboflow_dir = download_dirs[0] if download_dirs else ""

if roboflow_dir:
    print(f"Menggunakan direktori Roboflow: {roboflow_dir}")
    !python repo_ai/src/preprocessing/prepare_dataset.py --roboflow_dir "{roboflow_dir}" --output_dir "./data"
else:
    print("Folder dataset tidak ditemukan!")

## 5. Mulai Training! 🚀

Ini akan melatih model CLIP. Waktu yang dibutuhkan sekitar 20-40 menit tergantung jumlah dataset dan GPU (pastikan Runtime -> Change runtime type -> Hardware accelerator: **T4 GPU**).

In [ ]:
!python repo_ai/scripts/train.py --data_dir "./data" --output_dir "./models" --epochs 10 --batch_size 32

## 6. Unduh Hasil Model

Setelah training selesai, kita akan mengompres folder `models/` agar Anda bisa mengunduhnya ke laptop Anda.

In [ ]:
import shutil
from google.colab import files

# Zip folder models
shutil.make_archive("hasil_model_clip", 'zip', "./models")

# Download zip file
files.download("hasil_model_clip.zip")